### # Data Profiling – Transactions

This notebook profiles the transactions dataset to understand schema,
volume, time coverage, and key columns across all monthly files.




In [0]:
# Reading all the 24 files into a dataframe
df = (
    spark.read
    .option("header", True)
    .csv("/Volumes/workspace/default/coffee_raw_volume/transactions/")
)

df.printSchema()
df.show(5)


In [0]:
# Row count & uniqueness
from pyspark.sql.functions import count, countDistinct

df.select(
    count("*").alias("total_rows"),
    countDistinct("transaction_id").alias("distinct_transaction_ids")
).show()


In [0]:
# time Range analysis
from pyspark.sql.functions import min, max

df.select(
    min("created_at").alias("min_created_at"),
    max("created_at").alias("max_created_at")
).show()


In [0]:

# Amount Checks
df.select(
    "original_amount",
    "discount_applied",
    "final_amount"
).describe().show()


In [0]:
# Null checks
df.selectExpr(
    "sum(case when transaction_id is null then 1 else 0 end) as null_transaction_id",
    "sum(case when created_at is null then 1 else 0 end) as null_created_at",
    "sum(case when store_id is null then 1 else 0 end) as null_store_id",
    "sum(case when user_id is null then 1 else 0 end) as null_user_id"
).show()



###  Key Observations

- The transactions dataset represents one row per transaction and serves as the primary fact table.
- The dataset spans from July 2023 to June 2025 and is distributed across 24 monthly files.
- transaction_id is unique across the dataset, confirming correct transaction-level granularity.
- created_at is present for all records, enabling reliable time-based analysis and incremental processing.
- user_id is null for approximately 7.3 million transactions, indicating a significant number of guest or anonymous purchases.
- Amount-related columns (original_amount, discount_applied, final_amount) show valid and reasonable value ranges.

